# Telecom Churn Binary Classification

This notebook aims to build and train a deep neural network (DNN) using **TensorFlow/Keras** to predict customer churn (binary classification) based on various customer and service attributes from the provided telecom dataset.

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

2025-12-02 19:39:07.528344: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-02 19:39:07.528922: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-02 19:39:07.621555: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-02 19:39:08.885904: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

## Data Loading and Initial Exploration

We load the dataset and perform an initial inspection to understand its structure, data types, and identify missing values.

In [2]:
df = pd.read_csv("../data/telecom_churn_exam.csv")
df.head()

,Customer_ID,Name,Age,Yearly_Income,Plan_Type,Service_Rating,Churn
0,CUST_1000,Jane Smith,56.0,35795,Premium,3,0
1,CUST_1001,Bob Brown,36.0,148214,Basic,4,1
2,CUST_1002,Jane Rodriguez,NaN,80263,Standard,5,0
3,CUST_1003,David Smith,NaN,103104,Basic,4,1
4,CUST_1004,Bob Brown,61.0,113016,Basic,5,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Customer_ID     2000 non-null   object 
 1   Name            2000 non-null   object 
 2   Age             1815 non-null   float64
 3   Yearly_Income   2000 non-null   int64  
 4   Plan_Type       2000 non-null   object 
 5   Service_Rating  2000 non-null   int64  
 6   Churn           2000 non-null   int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 109.5+ KB


## Data Preprocessing

Based on the initial inspection, the following preprocessing steps are required:
1.  **Drop Irrelevant Columns:** `Customer_ID` and `Name` are identifiers and don't contribute to the prediction.
2.  **Handle Missing Values:** The `Age` column has missing values (`1815 non-null` out of `2000 entries`). We'll use the `dropna()` method to remove rows with NaN.
3.  **Feature Encoding:** The categorical feature `Plan_Type` needs to be converted into numerical format using **One-Hot Encoding**.

In [4]:
df = df.drop(["Name", "Customer_ID"], axis=1)
df.head()

,Age,Yearly_Income,Plan_Type,Service_Rating,Churn
0,56.0,35795,Premium,3,0
1,36.0,148214,Basic,4,1
2,NaN,80263,Standard,5,0
3,NaN,103104,Basic,4,1
4,61.0,113016,Basic,5,0


In [5]:
df.dropna(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1815 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             1815 non-null   float64
 1   Yearly_Income   1815 non-null   int64  
 2   Plan_Type       1815 non-null   object 
 3   Service_Rating  1815 non-null   int64  
 4   Churn           1815 non-null   int64  
dtypes: float64(1), int64(3), object(1)
memory usage: 85.1+ KB


In [6]:
# one-hot encode for the plan_type
df_encoded = df.join(pd.get_dummies(df.Plan_Type, dtype="int")).drop(["Plan_Type"], axis=1)
df_encoded.head()

,Age,Yearly_Income,Service_Rating,Churn,Basic,Premium,Standard
0,56.0,35795,3,0,0,1,0
1,36.0,148214,4,1,1,0,0
4,61.0,113016,5,0,1,0,0
5,33.0,79150,2,1,0,0,1
6,61.0,76886,5,0,0,1,0


## Feature Scaling and Data Splitting

1.  **Separate Features (X) and Target (y):** The target variable is `Churn`.
2.  **One-Hot Encoding for Target (y):** Convert the binary target (`0` or `1`) into a one-hot encoded format (`[1, 0]` or `[0, 1]`), which is necessary for the **`softmax` activation** in the final layer and the **`categorical_crossentropy` loss** function.
3.  **Scale Features:** Numerical features are scaled using `StandardScaler` to normalize them, which is crucial for training neural networks efficiently.
4.  **Train-Validation Split:** Split the data for training and evaluating the model.

In [7]:
# splitting the data
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder

X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

label_encoder = LabelEncoder()
y_encode = label_encoder.fit_transform(y)
y_onehot = tf.keras.utils.to_categorical(y_encode)

# scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# split the data into train and validate
X_train, X_valid, y_train, y_valid = train_test_split(X_scaled, y_onehot, test_size=0.4, random_state=42)

## Model Definition and Compilation

We define a Deep Neural Network (DNN) model and compile it for binary classification.

* **Architecture:** The model consists of 6 hidden layers, each with 64 neurons and `relu` activation. The output layer has **2 neurons** with `softmax` activation, matching the two classes in our one-hot encoded target. * **Compilation:**
    * **Optimizer:** `Adam` with a learning rate of $0.001$.
    * **Loss Function:** `categorical_crossentropy` (standard for multi-class classification, including binary classification with one-hot encoded labels).
    * **Metric:** `accuracy`.

In [8]:
model = keras.Sequential([
    keras.Input([X.shape[1]]),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(2, activation="softmax")
])

2025-12-02 19:39:10.004600: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [9]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

## Model Training

The model is trained for 100 epochs using a batch size of 120. We use **Early Stopping** to prevent overfitting, monitoring the validation loss (`val_loss`) and stopping if no improvement is seen after 3 epochs (`patience=3`).

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=120,
    validation_split=0.4,
    callbacks=[early_stop],
    verbose=2
)

Epoch 1/100
6/6 - 1s - 208ms/step - accuracy: 0.7167 - loss: 0.6545 - val_accuracy: 0.8647 - val_loss: 0.5911
Epoch 2/100
6/6 - 0s - 13ms/step - accuracy: 0.8836 - loss: 0.5329 - val_accuracy: 0.8899 - val_loss: 0.4485
Epoch 3/100
6/6 - 0s - 13ms/step - accuracy: 0.9142 - loss: 0.3764 - val_accuracy: 0.9014 - val_loss: 0.2882
Epoch 4/100
6/6 - 0s - 12ms/step - accuracy: 0.9250 - loss: 0.2370 - val_accuracy: 0.9174 - val_loss: 0.1867
Epoch 5/100
6/6 - 0s - 12ms/step - accuracy: 0.9311 - loss: 0.1677 - val_accuracy: 0.9472 - val_loss: 0.1388
Epoch 6/100
6/6 - 0s - 12ms/step - accuracy: 0.9464 - loss: 0.1483 - val_accuracy: 0.9404 - val_loss: 0.1299
Epoch 7/100
6/6 - 0s - 15ms/step - accuracy: 0.9495 - loss: 0.1361 - val_accuracy: 0.9495 - val_loss: 0.1283
Epoch 8/100
6/6 - 0s - 16ms/step - accuracy: 0.9525 - loss: 0.1328 - val_accuracy: 0.9404 - val_loss: 0.1269
Epoch 9/100
6/6 - 0s - 13ms/step - accuracy: 0.9510 - loss: 0.1294 - val_accuracy: 0.9495 - val_loss: 0.1263
Epoch 10/100
6/6 -

## Evaluation and Prediction

Finally, we evaluate the model's performance on the validation set.

In [11]:
# Evaluate
test_loss, test_acc = model.evaluate(X_valid, y_valid)
print(f"\nTest Accuracy: {test_acc:.3f}")

# Predict
predictions = model.predict(X_valid)
print(predictions[:5])
predicted_classes = np.argmax(predictions, axis=1)
actual_classes = np.argmax(y_valid, axis=1)

print("\nPredicted Classes:", predicted_classes[:10])
print("Actual Classes:   ", actual_classes[:10])

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9628 - loss: 0.1056 

Test Accuracy: 0.963
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
[[9.7494501e-01 2.5054971e-02]
 [7.7724320e-01 2.2275676e-01]
 [9.9994361e-01 5.6435525e-05]
 [9.9687248e-01 3.1275786e-03]
 [9.9884355e-01 1.1563918e-03]]

Predicted Classes: [0 0 0 0 0 1 1 1 0 1]
Actual Classes:    [0 0 0 0 0 1 1 1 0 1]
